::: {.sb-lesson-links}
[Exercises →](../exercises/titanic.ipynb)
[Quiz →](../quiz/quiz.qmd)
[All files of this section (zip)](../../files/06-ensemble-methods.zip)
:::

::: {.sb-downloads}
[Download this notebook](../downloads/lesson-boosting.ipynb){download="lesson-boosting.ipynb"}
:::

Boosting turns **weak** models (barely better than chance) into a **strong** one. The models are trained one after the other, and each new one focuses on what the previous ones got wrong. This lesson draws each equation, so you can see what happens behind it.

In [ ]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.datasets import make_classification, make_friedman1, make_moons
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from xgboost import XGBRegressor

# colour palette of the course
pio.templates["course"] = go.layout.Template(
    layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"]
)
pio.templates.default = "plotly_white+course"

## The idea

Preparing for a hard exam, you take a first mock exam. Next time you study, you spend more time on the questions you got wrong and less on the ones you got right. Boosting does the same: after each model, the observations it got wrong become heavier, so the next model pays more attention to them. In the end, all the models vote, and the better ones get a bigger say.

AdaBoost (Freund and Schapire, Gödel Prize 2003) was the first major boosting algorithm.

## AdaBoost

### The algorithm

Observations $X_1,\dots,X_n$, a target $Y_i\in\{-1,+1\}$.

**Start:** every observation gets the same weight, $D_1(i)=\frac{1}{n}$.

**For** $t=1,\dots,T$:

1. Train a weak model $h_t$ on the data weighted by $D_t$. It predicts $-1$ or $+1$.
2. **How wrong is it?** The weighted error (the weights sum to 1):
$$\epsilon_t=\sum_{i=1}^{n}D_t(i)\,\mathbb{1}\left[h_t(X_i)\neq Y_i\right]$$

3. **How much say does it get?**
$$\alpha_t=\frac{1}{2}\ln\left(\frac{1-\epsilon_t}{\epsilon_t}\right)$$

4. **Who needs more attention next time?**
$$D_{t+1}(i)=\frac{D_t(i)\,e^{-\alpha_t Y_i h_t(X_i)}}{Z_t}$$
where $Z_t$ makes the new weights sum to 1.

**Final model:** a weighted vote,
$$H(X)=\operatorname{sign}\Big(\underbrace{\sum_{t=1}^{T}\alpha_t\,h_t(X)}_{F(X)}\Big)$$

In step 4, $Y_i h_t(X_i)$ is $+1$ when the prediction is right and $-1$ when it is wrong. So a wrong point is multiplied by $e^{\alpha_t}$ and a right point by $e^{-\alpha_t}$.

### The equations, drawn

What do steps 3 and 4 really do? Let's plot them for every possible error $\epsilon_t$.

In [ ]:
eps = np.linspace(0.02, 0.98, 300)
alpha = 0.5 * np.log((1 - eps) / eps)
fig = make_subplots(rows=1, cols=2, subplot_titles=["Step 3: the say α of a model", "Step 4: weight multipliers"])
fig.add_trace(go.Scatter(x=eps, y=alpha, name="α = ½ ln((1-ε)/ε)"), row=1, col=1)
marks = np.array([0.1, 0.3, 0.5])
fig.add_trace(go.Scatter(x=marks, y=0.5 * np.log((1 - marks) / marks), mode="markers+text", showlegend=False,
                         text=[f"ε={e}: α={0.5 * np.log((1 - e) / e):.2f}" for e in marks], textposition="top right",
                         marker=dict(size=9, color="#111111")), row=1, col=1)
half = eps[eps <= 0.5]
alpha_half = 0.5 * np.log((1 - half) / half)
fig.add_trace(go.Scatter(x=half, y=np.exp(alpha_half), name="wrong points: × e^α"), row=1, col=2)
fig.add_trace(go.Scatter(x=half, y=np.exp(-alpha_half), name="right points: × e^-α"), row=1, col=2)
fig.add_hline(y=0, line_color="#999999", row=1, col=1)
fig.add_hline(y=1, line_dash="dash", line_color="#999999", row=1, col=2)
fig.update_xaxes(title_text="weighted error ε")
fig.update_yaxes(title_text="α", row=1, col=1)
fig.update_yaxes(title_text="multiplier (log scale)", type="log", row=1, col=2)
fig.update_layout(height=400)
fig.show()

**What this shows:**

* **Left:** a good model (small $\epsilon$) gets a big say: $\epsilon=0.1$ gives $\alpha=1.10$, $\epsilon=0.3$ gives $\alpha=0.42$. A coin flip ($\epsilon=0.5$) gets $\alpha=0$: it says nothing. A model worse than chance gets a negative $\alpha$: its vote is reversed.
* **Right:** the better the model, the harder the update. With $\epsilon=0.1$, the weight of each wrong point is multiplied by 3 and the weight of each right point divided by 3. With $\epsilon$ close to 0.5, weights barely move.

And here is what happens behind the normalization. Before dividing by $Z_t$, the wrong points weigh in total $\epsilon_t e^{\alpha_t}$ and the right points $(1-\epsilon_t)e^{-\alpha_t}$. With $e^{\alpha_t}=\sqrt{(1-\epsilon_t)/\epsilon_t}$, both are equal to $\sqrt{\epsilon_t(1-\epsilon_t)}$. So **after every update, the points the model got wrong hold exactly half of the total weight.** On the new weights, the last model is no better than a coin flip: the next one is forced to learn something new.

### See it with data 🔬: AdaBoost by hand

40 points in 2D, and the simplest weak model: a **decision stump** (a tree with a single split, so one straight cut). We code every step of the loop ourselves.

In [ ]:
X, y01 = make_moons(n_samples=40, noise=0.25, random_state=0)
y = np.where(y01 == 1, 1, -1)                        # the classes must be -1 and +1

n = len(y)
D = np.full(n, 1 / n)                                # start: same weight for everyone
stumps, alphas, weights, rounds = [], [], [], []
for t in range(1, 51):
    stump = DecisionTreeClassifier(max_depth=1).fit(X, y, sample_weight=D)   # 1. weak model h_t
    h = stump.predict(X)
    eps = D[h != y].sum()                            # 2. weighted error
    alpha = 0.5 * np.log((1 - eps) / eps)            # 3. say of this model
    weights.append(D)                                # (the weights this stump was trained on)
    D = D * np.exp(-alpha * y * h)                   # 4. wrong points x e^alpha, right points x e^-alpha
    D = D / D.sum()                                  #    normalization (Z_t)
    stumps.append(stump)
    alphas.append(alpha)
    rounds.append({"round": t, "wrong points": int((h != y).sum()), "ε": round(eps, 3), "α": round(alpha, 3),
                   "wrong points × e^α": round(np.exp(alpha), 2), "right points × e^-α": round(np.exp(-alpha), 2),
                   "weight held by the wrong points after the update": round(D[h != y].sum(), 3)})
pd.DataFrame(rounds).head(4)

,round,wrong points,ε,α,wrong points × e^α,right points × e^-α,weight held by the wrong points after the update
0,1,10,0.250,0.549,1.73,0.58,0.5
1,2,12,0.200,0.693,2.00,0.50,0.5
2,3,12,0.281,0.469,1.60,0.63,0.5
3,4,10,0.217,0.640,1.90,0.53,0.5


In [ ]:
fig = make_subplots(rows=2, cols=2, vertical_spacing=0.12, horizontal_spacing=0.06,
                    subplot_titles=[f"Round {t}: ε = {rounds[t - 1]['ε']}, α = {rounds[t - 1]['α']}" for t in range(1, 5)])
for t in range(4):
    h = stumps[t].predict(X)
    fig.add_trace(go.Scatter(
        x=X[:, 0], y=X[:, 1], mode="markers", showlegend=False,
        marker=dict(size=4 + 400 * weights[t], color=np.where(y == 1, "#3B47C4", "#E4572E"),
                    symbol=np.where(h != y, "x", "circle"), line=dict(width=1, color="white")),
        hovertext=[f"weight = {w:.3f}" for w in weights[t]]), row=t // 2 + 1, col=t % 2 + 1)
    feature, threshold = stumps[t].tree_.feature[0], stumps[t].tree_.threshold[0]
    if feature == 0:
        fig.add_vline(x=threshold, line_width=2, row=t // 2 + 1, col=t % 2 + 1)
    else:
        fig.add_hline(y=threshold, line_width=2, row=t // 2 + 1, col=t % 2 + 1)
fig.update_layout(title="Size = weight the stump was trained on · line = its cut · ✕ = misclassified", height=720)
fig.show()

**What this shows:** in round 1 every point weighs 1/40 and the stump cuts at $x_1 = 1.29$, with 10 mistakes ($\epsilon_1 = 0.25$, $\alpha_1 = 0.55$). Those 10 points then hold half the weight, and they grow on the next panel. The round 2 stump changes direction (a horizontal cut) to get them right. Each round, the ✕ points of one panel become the big points of the next. The last column of the table confirms the property above: after each update the misclassified points hold exactly 0.5 of the weight.

### See it with data 🔬: the final vote $F(X)=\sum_t \alpha_t h_t(X)$, drawn

Each stump adds $+\alpha_t$ on one side of its cut and $-\alpha_t$ on the other. Let's colour the plane with $F$ after 1, 3, 10 and 50 rounds: blue where $F>0$ (class +1), orange where $F<0$.

In [ ]:
grid1, grid2 = np.meshgrid(np.linspace(-1.6, 2.6, 200), np.linspace(-1.3, 1.8, 200))
grid = np.c_[grid1.ravel(), grid2.ravel()]
fig = make_subplots(rows=2, cols=2, vertical_spacing=0.12, horizontal_spacing=0.06, subplot_titles=[f"T = {T}" for T in [1, 3, 10, 50]])
for k, T in enumerate([1, 3, 10, 50]):
    F_grid = sum(a * s.predict(grid) for a, s in zip(alphas[:T], stumps[:T]))
    F_train = sum(a * s.predict(X) for a, s in zip(alphas[:T], stumps[:T]))
    fig.layout.annotations[k].text = f"{T} stump{'s' if T > 1 else ''}: train accuracy {(np.sign(F_train) == y).mean():.3f}"
    fig.add_trace(go.Heatmap(x=grid1[0], y=grid2[:, 0], z=F_grid.reshape(grid1.shape), zmid=0, showscale=False,
                             colorscale=[[0, "#E4572E"], [0.5, "#FFFFFF"], [1, "#3B47C4"]], opacity=0.6,
                             hoverinfo="skip"), row=k // 2 + 1, col=k % 2 + 1)
    fig.add_trace(go.Scatter(x=X[:, 0], y=X[:, 1], mode="markers", showlegend=False,
                             marker=dict(size=6, color=np.where(y == 1, "#3B47C4", "#E4572E"),
                                         line=dict(width=1, color="#111111"))), row=k // 2 + 1, col=k % 2 + 1)
fig.update_layout(title="F(X): darker = a larger majority in the vote", height=720)
fig.show()

ada = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=50, random_state=0).fit(X, y)
F_grid = sum(a * s.predict(grid) for a, s in zip(alphas, stumps))
print("Same predictions as scikit-learn's AdaBoostClassifier:", (ada.predict(grid) == np.sign(F_grid)).all())

Same predictions as scikit-learn's AdaBoostClassifier: True


**What this shows:** one stump is one straight cut (75% train accuracy). With 3 stumps, the cuts add up into a staircase (87.5%). With 50, the staircase follows the two moons and every training point is on the right side (100%). The colour intensity is $|F|$: deep colour where the stumps agree, pale near the border where the vote is close. Scikit-learn's `AdaBoostClassifier` gives exactly the same predictions: this little loop *is* AdaBoost.

### The weak learning condition

Each model must do better than chance on its weighted data: $\epsilon_t<\frac{1}{2}-\gamma$ with $\gamma>0$ (with unbalanced classes, "chance" means the baseline error). If $\epsilon_t$ reaches 0.5, the say $\alpha_t$ drops to 0 and boosting stops making progress.

### See it with data 🔬: one stump vs 50 stumps

1,000 noisy moons, 30% kept aside as a test set.

In [ ]:
X, y = make_moons(n_samples=1000, noise=0.3, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

stump = DecisionTreeClassifier(max_depth=1).fit(X_train, y_train)
ada = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=50, random_state=0).fit(X_train, y_train)

print(f"One stump            : test accuracy = {stump.score(X_test, y_test):.3f}")
print(f"AdaBoost of 50 stumps: test accuracy = {ada.score(X_test, y_test):.3f}")
errors = ada.estimator_errors_                       # weighted error ε_t of each stump, on its own weighting
print(f"Weighted error of the 50 stumps: from {errors.min():.3f} to {errors.max():.3f}")

One stump            : test accuracy = 0.773
AdaBoost of 50 stumps: test accuracy = 0.890
Weighted error of the 50 stumps: from 0.181 to 0.485


In [ ]:
fig = go.Figure(go.Scatter(x=np.arange(1, len(errors) + 1), y=errors, mode="lines+markers", name="ε_t"))
fig.add_hline(y=0.5, line_dash="dash", annotation_text="0.5: no better than chance")
fig.update_layout(title="Weighted error of each stump during boosting", xaxis_title="round t",
                  yaxis_title="weighted error ε_t", yaxis_range=[0, 0.55])
fig.show()

**What this shows:** one stump reaches 0.773 test accuracy, 50 stumps together 0.890. Every stump stays below 0.5, but its error climbs over the rounds: the easy points are already handled, the hard ones carry most of the weight, so each new stump has less to say ($\alpha_t$ shrinks).

### Overfitting and margins

![Train and test error of AdaBoost as boosting rounds are added](../img/boosting_rounds.png)

AdaBoost brings the training error down fast. With noisy data, it can overfit: it keeps putting weight on points it gets wrong, even when they are just mislabelled noise.

But often the test error keeps going down **after** the training error has reached 0. The explanation is the **margin** of an observation, the strength of the majority in the vote:

$$\text{margin}(X_i)=\frac{Y_i\,F(X_i)}{\sum_t \alpha_t}\in[-1,1]$$

$+1$: every model is right; $0$: a tie; negative: the final vote is wrong. Extra rounds keep pushing the small margins up, which makes the predictions on new data more reliable.

![AdaBoost: train and test error over 1,000 rounds, and the distribution of the training margins](../img/boosting_margins.png)

On this simulated dataset, the training error is 0 after about 50 rounds (left), yet the test error keeps falling from 14% to 12%. On the right, the smallest training margins keep growing between 5, 100 and 1,000 rounds.

## Gradient boosting (XGBoost)



::: {.callout-tip title="Intuition"}
Think of a golfer. The second shot does not start again from the tee: it starts from where the ball is and only covers the remaining distance. AdaBoost *re-weights* the observations it gets wrong; gradient boosting trains the next model directly on *what is still missing*.
:::



Start with a first model $F_1$. Its residuals are $\epsilon_1=y-F_1(X)$. Train a new model $f_1$ to predict these residuals, and add it:

$$F_{i+1}=F_i+\nu\,f_i,\qquad f_i \text{ trained on } y-F_i(X)$$

$\nu$ is the **learning rate** (between 0 and 1): each tree corrects only part of the error.

Why "gradient"? With the squared error $L=\frac{1}{2}(y-F)^2$, the gradient with respect to the prediction is $-\frac{\partial L}{\partial F}=y-F$: the residual. Each tree takes one step of gradient descent, in the space of predictions. With other losses (log-loss for classification...), the trees fit the gradient of that loss instead.

### See it with data 🔬: gradient boosting by hand

80 points of a noisy wavy curve. $F_1$ is the mean of $y$; each step trains a tree of depth 2 on the residuals and adds it with a learning rate of 0.5.

In [ ]:
rng = np.random.default_rng(0)
x = np.sort(rng.uniform(0, 10, 80))
y = 3 * np.sin(x) + x / 2 + rng.normal(0, 0.5, 80)
X = x.reshape(-1, 1)

learning_rate = 0.5
pred = np.full_like(y, y.mean())                         # F_1: predict the mean for everyone
steps = [pred.copy()]
for i in range(3):
    residuals = y - pred                                 # what is still missing
    f = DecisionTreeRegressor(max_depth=2).fit(X, residuals)   # f_i is trained on the residuals
    pred = pred + learning_rate * f.predict(X)           # F_(i+1) = F_i + learning_rate * f_i
    steps.append(pred.copy())
for k, p in enumerate(steps):
    print(f"F_{k + 1}: mean absolute residual = {np.abs(y - p).mean():.3f}")

F_1: mean absolute residual = 1.972
F_2: mean absolute residual = 1.295
F_3: mean absolute residual = 0.917
F_4: mean absolute residual = 0.740


In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Prediction after each step", "Residuals after each step"])
fig.add_trace(go.Scatter(x=x, y=y, mode="markers", name="data", marker=dict(color="#999999")), row=1, col=1)
colors = ["#E4572E", "#F2C14E", "#4BA3C3", "#3B47C4"]
for k, p in enumerate(steps):
    fig.add_trace(go.Scatter(x=x, y=p, mode="lines", name=f"F_{k + 1}", line=dict(shape="hv", color=colors[k])), row=1, col=1)
    fig.add_trace(go.Box(y=y - p, name=f"F_{k + 1}", marker_color=colors[k], showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="x", row=1, col=1)
fig.update_yaxes(title_text="y", row=1, col=1)
fig.update_yaxes(title_text="residual y - F(x)", row=1, col=2)
fig.update_layout(title="Gradient boosting: each tree corrects part of what is still missing", height=420)
fig.show()

In [ ]:
gb = GradientBoostingRegressor(n_estimators=3, learning_rate=0.5, max_depth=2).fit(X, y)
print("Same predictions as scikit-learn's GradientBoostingRegressor:", np.allclose(gb.predict(X), pred))

Same predictions as scikit-learn's GradientBoostingRegressor: True


**What this shows:** the mean alone is off by 1.972 on average, then 1.295, 0.917 and 0.740 after three trees. On the left the flat line becomes a staircase that follows the wave; on the right the residuals shrink around 0. Scikit-learn's `GradientBoostingRegressor` gives exactly the same predictions.

### See it with data 🔬: learning rate vs number of trees

`make_friedman1`: 2,000 observations, 10 features (5 useful), noise. 4 learning rates, up to 300 trees of depth 3.

In [ ]:
X, y = make_friedman1(n_samples=2000, noise=1.0, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

curves = {}
for lr in [1.0, 0.3, 0.1, 0.03]:
    gb = GradientBoostingRegressor(learning_rate=lr, n_estimators=300, max_depth=3, random_state=0)
    gb.fit(X_train, y_train)
    curves[lr] = [r2_score(y_test, p) for p in gb.staged_predict(X_test)]   # test R² after 1, 2, ..., 300 trees

table = pd.DataFrame({f"{n} trees": {lr: c[n - 1] for lr, c in curves.items()} for n in [10, 30, 100, 300]})
table.rename_axis("learning rate").round(3)

,10 trees,30 trees,100 trees,300 trees
learning rate,,,,
1.00,0.831,0.849,0.823,0.813
0.30,0.814,0.909,0.918,0.918
0.10,0.578,0.815,0.917,0.929
0.03,0.274,0.543,0.814,0.914


In [ ]:
fig = go.Figure()
for lr, c in curves.items():
    fig.add_trace(go.Scatter(x=np.arange(1, len(c) + 1), y=c, mode="lines", name=f"learning rate = {lr}"))
fig.update_layout(title="Gradient boosting: test R² as trees are added", xaxis_title="number of trees",
                  yaxis_title="test R²", yaxis_range=[0, 1])
fig.show()

**What this shows:** with a learning rate of 1 the model is fast (0.831 after 10 trees) but peaks early and overfits (0.813 after 300). With 0.03 it is very slow (0.274 after 10 trees) and only gets good at the end (0.914). The best score, 0.929, comes from 0.1 with 300 trees. **The smaller the learning rate, the more trees you need**: tune them together.

### See it with data 🔬: XGBoost

XGBoost ("eXtreme Gradient Boosting") implements the same idea, with regularized trees and a fast, parallel implementation. Same settings for both: 300 trees of depth 3, learning rate 0.1.

In [ ]:
models = {"scikit-learn GradientBoosting": GradientBoostingRegressor(n_estimators=300, learning_rate=0.1, max_depth=3, random_state=0),
          "XGBoost": XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=3, random_state=0, n_jobs=2)}  # 2 CPU cores
XGBRegressor(n_estimators=1).fit(X_train[:50], y_train[:50])   # warm-up: the very first call loads the library

for name, model in models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    duration = time.perf_counter() - start
    print(f"{name:<30} test R² = {model.score(X_test, y_test):.3f}   fit time = {duration:.2f} s")

scikit-learn GradientBoosting  test R² = 0.929   fit time = 2.23 s


XGBoost                        test R² = 0.937   fit time = 0.25 s


**What this shows:** similar scores (test $R^2$ = 0.929 for scikit-learn, 0.937 for XGBoost), but XGBoost trains several times faster (the times depend on your computer). On millions of rows, this is why XGBoost and its cousins (LightGBM, CatBoost, scikit-learn's `HistGradientBoostingRegressor`) are the usual choice.

::: {.sb-next}
**[Next: Voting and stacking →](../lessons/voting-and-stacking.ipynb)**
:::